# `sim.signals`: quantities that change over time
**Host only.** The simulation never runs on the board, so this notebook has no hardware
section. Every cell runs in ordinary Python (CPython) on this computer.

# TODO:
Nothing open.

Decided (2026-11-16): a signal is any callable `f(t_s) -> value`. A fake's physical quantity
(temperature, humidity, field, distance) is either a plain value or a signal, and the fake
reads it through `value()` at the moment the driver asks. Coupling between parts is a closure:
`temp_c=lambda t: 30 if relay.closed else 20`. No event loop and no registration.

# Python file
location: `src/micropython_i2c/sim/signals.py`

tests: `tests/test_sim_signals.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/sim/signals.py
"""Signals: physical quantities that change over (simulated) time, for the fakes.

A signal is any callable ``f(t_s)`` returning a value at time ``t_s`` seconds. A fake's quantity
is a plain value or a signal; the fake reads it with ``value(x)``, which calls a signal at the
current ``compat`` time (``SimClock`` time inside a ``with SimClock()`` block) and returns a
plain value unchanged. Tuples and lists are resolved item by item, so a 3-axis field can be
``(ramp(...), 0.0, sine(...))``.

The builders here cover the common shapes; any ``lambda t: ...`` works too, including one that
reads another fake (``lambda t: 30.0 if relay.closed else 20.0``).

Host only: uses CPython's ``math`` and ``random``.

Generated by ``dev_notebooks/sim_signals_devNB``: edit the notebook, not this file.
"""

# Imports
## External
`math` for `sine` and `lag`, `random` for `noisy` (seeded, so a run repeats).

## Internal
`compat.ticks_us` for the current time on whichever clock is installed.

In [ ]:
%%extract ../src/micropython_i2c/sim/signals.py -a
import math
import random

from micropython_i2c.compat import ticks_us

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_sim_signals.py
import pytest

from micropython_i2c.sim.clock import SimClock
from micropython_i2c.sim.signals import lag, noisy, now_s, ramp, sequence, sine, step, value

Notebook-only: the imports the validation and test cells use.

In [ ]:
import pytest

from micropython_i2c.sim.clock import SimClock

# Intent
**Purpose:** give the fakes scenarios. Without signals a fake reports whatever constant a test
set; with them, "the room warms from 20 C to 26 C over an hour", "a hand comes within 80 mm at
t = 5 s" or "the heater follows the relay with a 2-minute lag" are one line each.

| Builder | Shape |
|---|---|
| `ramp(a, b, t0, t1)` | `a` until `t0`, straight line to `b` at `t1`, then `b` |
| `step(before, after, at)` | `before`, then `after` from `at` on |
| `sine(mean, amplitude, period)` | a sine wave around `mean` |
| `sequence([(t, v), ...])` | straight lines through the points, flat before the first and after the last |
| `noisy(x, sigma, seed)` | `x` plus Gaussian noise, seeded so a run repeats |
| `lag(target, tau)` | follows `target` with a first-order lag of time constant `tau` |

Arguments of the builders may themselves be signals or plain values: `noisy(ramp(...), 0.1)`
and `lag(lambda t: 30 if relay.closed else 20, 120)` both work.

```python
with SimClock() as clock:
    part = FakeSHTC3(temp_c=ramp(20.0, 26.0, 0, 3600))
    clock.advance(1800)
    sht.measure()           # (23.0..., 50.0)
```

**Not its job:** physics. A signal is a description of what the quantity does, not a model
that works it out.

## What the module defines

| Name | What it is |
|---|---|
| `now_s` | the current time in seconds, on whichever clock `compat` uses |
| `value` | resolve a plain value, a signal, or a tuple/list of either, at `now_s()` |
| `ramp`, `step`, `sine`, `sequence`, `noisy`, `lag` | signal builders |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/sim/signals.py -a
__all__ = ["lag", "noisy", "now_s", "ramp", "sequence", "sine", "step", "value"]

## `now_s` and `value`
`now_s` reads the installed clock, so a signal sees simulated time inside `with SimClock()`
and real time (seconds since import) outside. `value` is what every fake calls: it is the one
place that knows a quantity may be a signal.

In [ ]:
%%extract ../src/micropython_i2c/sim/signals.py -a
def now_s():
    """The current time in seconds, on the clock ``compat`` uses."""
    return ticks_us() / 1_000_000


def value(x, t_s=None):
    """``x`` at time ``t_s`` (default ``now_s()``).

    Args:
        x: A plain value, a signal (any callable ``f(t_s)``), or a tuple/list of either.
        t_s: The time to evaluate at; ``None`` means now.

    Returns:
        ``x`` itself if it is a plain value, ``x(t_s)`` if it is a signal, and a tuple/list of
        the resolved items if it is a tuple/list.
    """
    if t_s is None:
        t_s = now_s()
    if callable(x):
        return x(t_s)
    if isinstance(x, (tuple, list)):
        return type(x)(value(item, t_s) for item in x)
    return x

### validation/test

#### validation: nominal usage

In [ ]:
with SimClock(start_s=3):
    print("plain:", value(21.5), "| signal:", value(lambda t: 2 * t), "| tuple:", value((1.0, lambda t: t, 0)))
    print("at an explicit time:", value(lambda t: 2 * t, t_s=10))

#### validation: anti-nominal usage
None: `value` accepts anything. A signal that raises raises through `value` unchanged.

#### unit

In [ ]:
%%extract ../tests/test_sim_signals.py -a --strip-ipytest
%%ipytest


def test_value_resolves_plain_values_signals_and_tuples():
    with SimClock(start_s=4):
        assert now_s() == 4.0
        assert value(7) == 7
        assert value(lambda t: t * 10) == 40.0
        assert value((1, lambda t: t)) == (1, 4.0)
        assert value([lambda t: -t]) == [-4.0]
        assert value(lambda t: t, t_s=1.5) == 1.5

## Shapes: `ramp`, `step`, `sine`, `sequence`
Pure functions of time. Their arguments may be signals too, resolved at the same instant.

In [ ]:
%%extract ../src/micropython_i2c/sim/signals.py -a
def ramp(start, end, t0_s, t1_s):
    """``start`` until ``t0_s``, a straight line to ``end`` at ``t1_s``, then ``end``.

    Raises:
        ValueError: If ``t1_s`` is not after ``t0_s``.
    """
    if t1_s <= t0_s:
        raise ValueError("ramp: t1_s ({}) must be after t0_s ({})".format(t1_s, t0_s))

    def signal(t):
        a, b = value(start, t), value(end, t)
        if t <= t0_s:
            return a
        if t >= t1_s:
            return b
        return a + (b - a) * (t - t0_s) / (t1_s - t0_s)

    return signal


def step(before, after, at_s):
    """``before`` until ``at_s``, then ``after``."""

    def signal(t):
        return value(after if t >= at_s else before, t)

    return signal


def sine(mean, amplitude, period_s, phase_s=0.0):
    """``mean + amplitude * sin(2 pi (t - phase_s) / period_s)``.

    Raises:
        ValueError: If ``period_s`` is not positive.
    """
    if period_s <= 0:
        raise ValueError("sine: period_s must be positive, got {}".format(period_s))

    def signal(t):
        return value(mean, t) + value(amplitude, t) * math.sin(
            2 * math.pi * (t - phase_s) / period_s
        )

    return signal


def sequence(points):
    """Straight lines through ``[(t_s, value), ...]``; flat before the first and after the last.

    Raises:
        ValueError: If there are no points, or the times do not increase.
    """
    pts = list(points)
    if not pts:
        raise ValueError("sequence: needs at least one (t_s, value) point")
    for (ta, _), (tb, _) in zip(pts, pts[1:]):
        if tb <= ta:
            raise ValueError("sequence: times must increase, got {} then {}".format(ta, tb))

    def signal(t):
        if t <= pts[0][0]:
            return value(pts[0][1], t)
        for (ta, va), (tb, vb) in zip(pts, pts[1:]):
            if t <= tb:
                a, b = value(va, t), value(vb, t)
                return a + (b - a) * (t - ta) / (tb - ta)
        return value(pts[-1][1], t)

    return signal

### validation/test

#### validation: nominal usage
One row per second for a few seconds of each shape.

In [ ]:
shapes = {
    "ramp(20, 26, 1, 4)": ramp(20.0, 26.0, 1, 4),
    "step(0, 1, 2)": step(0, 1, 2),
    "sine(10, 2, 4)": sine(10.0, 2.0, 4),
    "sequence": sequence([(0, 0.0), (2, 10.0), (3, 4.0)]),
}
print("t   " + "  ".join("{:>18}".format(k) for k in shapes))
for t in range(6):
    print("{:<3} ".format(t) + "  ".join("{:>18.3f}".format(value(s, t)) for s in shapes.values()))

#### validation: anti-nominal usage

In [ ]:
try:
    ramp(0, 1, 5, 5)  # t1 must be after t0
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("ramp with t1 == t0 should have raised ValueError")

In [ ]:
try:
    sequence([(0, 1), (0, 2)])  # times must increase
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("sequence with repeated times should have raised ValueError")

In [ ]:
try:
    sine(0, 1, 0)  # a period must be positive
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("sine with period 0 should have raised ValueError")

#### unit

In [ ]:
%%extract ../tests/test_sim_signals.py -a --strip-ipytest
%%ipytest


def test_ramp_is_flat_then_linear_then_flat():
    r = ramp(20.0, 26.0, 10, 70)
    assert value(r, 0) == 20.0
    assert value(r, 40) == pytest.approx(23.0)
    assert value(r, 100) == 26.0

In [ ]:
%%extract ../tests/test_sim_signals.py -a --strip-ipytest
%%ipytest


def test_step_switches_at_its_time():
    s = step("off", "on", 5)
    assert [value(s, t) for t in (4.999, 5, 6)] == ["off", "on", "on"]

In [ ]:
%%extract ../tests/test_sim_signals.py -a --strip-ipytest
%%ipytest


def test_sine_period_and_amplitude():
    s = sine(10.0, 2.0, 8)
    assert value(s, 0) == pytest.approx(10.0)
    assert value(s, 2) == pytest.approx(12.0)
    assert value(s, 6) == pytest.approx(8.0)
    assert value(s, 8) == pytest.approx(10.0, abs=1e-9)

In [ ]:
%%extract ../tests/test_sim_signals.py -a --strip-ipytest
%%ipytest


def test_sequence_interpolates_and_holds_its_ends():
    s = sequence([(1, 0.0), (3, 10.0), (4, 4.0)])
    assert [value(s, t) for t in (0, 1, 2, 3, 3.5, 9)] == [0.0, 0.0, 5.0, 10.0, 7.0, 4.0]

In [ ]:
%%extract ../tests/test_sim_signals.py -a --strip-ipytest
%%ipytest


def test_builders_accept_signals_as_arguments():
    r = ramp(lambda t: t, 100.0, 10, 20)  # the start moves with time
    assert value(r, 5) == 5.0
    assert value(r, 15) == pytest.approx(57.5)

In [ ]:
%%extract ../tests/test_sim_signals.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize(
    "build, match",
    [
        (lambda: ramp(0, 1, 2, 1), "after"),
        (lambda: sine(0, 1, -1), "positive"),
        (lambda: sequence([]), "at least one"),
        (lambda: sequence([(2, 0), (1, 0)]), "increase"),
    ],
)
def test_bad_shapes_are_refused(build, match):
    with pytest.raises(ValueError, match=match):
        build()

## Stateful signals: `noisy` and `lag`
`noisy` draws a fresh Gaussian sample on every read (a sensor's noise is per measurement), from
its own seeded generator so a run repeats. `lag` is a first-order follower: between two reads
`dt` apart it moves the fraction $1 - e^{-dt/\tau}$ of the way to the target. It works out the
whole gap at once, so it is exact for a target that held still in between however rarely it is
read. Reading at an earlier time than the last read (a new clock) starts it over.

In [ ]:
%%extract ../src/micropython_i2c/sim/signals.py -a
def noisy(signal, sigma, seed=0):
    """``signal`` plus Gaussian noise of standard deviation ``sigma``, seeded."""
    rng = random.Random(seed)

    def noisy_signal(t):
        return value(signal, t) + rng.gauss(0.0, value(sigma, t))

    return noisy_signal


def lag(target, tau_s, initial=None):
    """Follow ``target`` with a first-order lag of time constant ``tau_s`` seconds.

    Args:
        target: The value to follow (a plain value or a signal).
        tau_s: Time constant: after ``tau_s`` the output has covered 63 % of a step.
        initial: The output at the first read; default the target's value then.

    Raises:
        ValueError: If ``tau_s`` is not positive.
    """
    if tau_s <= 0:
        raise ValueError("lag: tau_s must be positive, got {}".format(tau_s))
    state = {"t": None, "y": None}

    def lagged(t):
        u = value(target, t)
        if state["t"] is None or t < state["t"]:  # first read, or a new clock: start over
            state["t"] = t
            state["y"] = u if initial is None else initial
            return state["y"]
        dt = t - state["t"]
        state["y"] += (u - state["y"]) * (1 - math.exp(-dt / tau_s))
        state["t"] = t
        return state["y"]

    return lagged

### validation/test

#### validation: nominal usage
A heater: the target jumps from 20 C to 30 C at t = 0, and the output follows with a 60 s
time constant (63 % at 60 s, 95 % at 180 s). Then noise around a constant, repeatable by seed.

In [ ]:
heat = lag(step(20.0, 30.0, 0), 60, initial=20.0)
for t in (0, 30, 60, 120, 180, 600):
    print("t = {:>3} s: {:6.2f} C".format(t, heat(t)))
n1, n2 = noisy(25.0, 0.1, seed=7), noisy(25.0, 0.1, seed=7)
print("same seed, same noise:", [round(n1(0), 4) for _ in range(3)] == [round(n2(0), 4) for _ in range(3)])

#### validation: anti-nominal usage

In [ ]:
try:
    lag(20.0, 0)  # a time constant must be positive
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("lag with tau 0 should have raised ValueError")

#### unit

In [ ]:
%%extract ../tests/test_sim_signals.py -a --strip-ipytest
%%ipytest


def test_lag_covers_63_percent_in_one_time_constant():
    heat = lag(step(20.0, 30.0, 0), 60, initial=20.0)
    assert heat(0) == 20.0
    assert heat(60) == pytest.approx(20.0 + 10.0 * (1 - 2.718281828 ** -1), abs=1e-6)

In [ ]:
%%extract ../tests/test_sim_signals.py -a --strip-ipytest
%%ipytest


def test_lag_result_does_not_depend_on_how_often_it_is_read():
    rare = lag(30.0, 10, initial=20.0)
    often = lag(30.0, 10, initial=20.0)
    rare(0)
    often(0)
    for t in range(1, 21):
        often(t)
    assert rare(20) == pytest.approx(often(20))

In [ ]:
%%extract ../tests/test_sim_signals.py -a --strip-ipytest
%%ipytest


def test_lag_starts_over_when_time_goes_back():
    f = lag(30.0, 10, initial=20.0)
    f(100)
    f(200)
    assert f(0) == 20.0

In [ ]:
%%extract ../tests/test_sim_signals.py -a --strip-ipytest
%%ipytest


def test_noise_has_the_right_spread_and_repeats_by_seed():
    a = noisy(25.0, 0.5, seed=3)
    samples = [a(0) for _ in range(4000)]
    mean = sum(samples) / len(samples)
    sd = (sum((s - mean) ** 2 for s in samples) / len(samples)) ** 0.5
    assert mean == pytest.approx(25.0, abs=0.05)
    assert sd == pytest.approx(0.5, abs=0.05)
    b = noisy(25.0, 0.5, seed=3)
    assert [b(0) for _ in range(5)] == samples[:5]

In [ ]:
%%extract ../tests/test_sim_signals.py -a --strip-ipytest
%%ipytest


def test_signals_follow_the_simulated_clock():
    r = ramp(0.0, 100.0, 0, 100)
    with SimClock() as clock:
        assert value(r) == 0.0
        clock.advance(25)
        assert value(r) == 25.0